In [0]:
class BasePreprocessor(ABC):
    @abstractmethod
    def process(self):
        """
        Returns
        -------
        pyspark.sql.DataFrame
            Processed dataframe.
        """
        pass

In [0]:
class SalesProcess(BasePreprocessor):
    def __init__(self,raw_data,filter_map):
        self.raw_data = raw_data
        self.filter_map = filter_map
        self.raw_sales = self.raw_data['raw_sales']
        self.calendar_data = self.raw_data['calendar']
    
    def unpivot_sales_data(self):
        self.sales_long = self.raw_sales\
                            .unpivot(ids=self.filter_map['sales_id_cols'],
                                    values = [c for c in self.raw_sales.columns if c.startswith('d_')],
                                            variableColumnName = "d",
                                            valueColumnName = "sales")
                            
        self.sales_long = self.sales_long.withColumn('id',F.regexp_replace(F.col('id'),'_evaluation',""))
        return self.sales_long
    
    def join_sales_calndar(self):
        self.calendar_filtered = self.calendar_data.select('d','date').distinct()
        self.sales_processed = self.sales_long.join(self.calendar_filtered,on=['d'],how='left')
        return self.sales_processed
    
    def process(self):
        self.sales_long = self.unpivot_sales_data()
        self.sales_processed = self.join_sales_calndar()
        return self.sales_processed

In [0]:
class EventsProcess(BasePreprocessor):
    def __init__(self,raw_data,filter_map):
        self.raw_data = raw_data
        self.filter_map = filter_map
        self.events_data = self.raw_data['calendar'].select('d','date','year','event_name_1','event_name_2','snap_CA','snap_WI','snap_TX').distinct()
        self.event_flag_cond = ((F.col('event_name_1').isNotNull())|(F.col('event_name_2').isNotNull()))

    def normalize_event_name(self, event):
        return (
            event.lower()
            .replace(" ", "_")
            .replace("'", "")
            .replace("-", "_")
        )

    def encode_events(self,events_data):  
        events_data = events_data.withColumn('event_flag',
                                             F.when(self.event_flag_cond,F.lit(1)).otherwise(F.lit(0)))\
                            .withColumn('event_flag',F.col('event_flag').cast('byte'))
        event1_values = [r["event_name_1"] for r in events_data.select("event_name_1").distinct().collect() if r["event_name_1"] is not None]
        event2_values = [r["event_name_2"] for r in events_data.select("event_name_2").distinct().collect() if r["event_name_2"] is not None]

        all_events = set(event1_values) | set(event2_values)

        event_exprs = {}
        for event in all_events:
            normalized_event = self.normalize_event_name(event)
            event_exprs[f"f_event_{normalized_event}"] = F.when(
                (F.col("event_name_1") == event) |
                (F.col("event_name_2") == event),
                1
            ).otherwise(0)

        events_encoded = events_data.withColumns(event_exprs)
        
        return events_encoded
    
    def process(self):
        self.events_processed = self.encode_events(self.events_data)
        return self.events_processed


    


In [0]:
class PriceProcess(BasePreprocessor):
    def __init__(self,raw_data,filter_map):
        self.raw_data = raw_data
        self.filter_map = filter_map
        self.price_data = self.raw_data['raw_price']
        self.calendar = self.raw_data['calendar'].select('d','date','wm_yr_wk').distinct()

    def process(self):
        processed_price = self.price_data.join(self.calendar,on=['wm_yr_wk'],how='left').select('store_id','item_id','sell_price','d','date')
        return processed_price

In [0]:
class JoinedDataProcess(BasePreprocessor):
    def __init__(self,filter_map,sales,events,price):
        self.filter_map = filter_map
        self.sales = sales
        self.events = events
        self.price = price
    
    def transform_data(self):
        joined_data = self.sales.join(self.events,on=['d','date'],how='left')
        joined_data = joined_data.join(self.price,on=['d','date','store_id','item_id'],how='left')\
                                            .withColumn('week',F.weekofyear(F.col('date')))\
                                            .withColumn('week',F.lpad(F.col('week'),2,'0'))\
                                            .withColumn('yearweek',F.concat(F.col('year'),F.col('week')))

        return joined_data
    
    def agg_data(self,joined_data):

        group_cols = ['id','item_id','dept_id','cat_id','store_id','state_id','yearweek']
        
        event_cols = [c for c in joined_data.columns if c.startswith('f_event_')]
        
        event_aggs = [F.max(F.col(c).cast('byte')).alias(c) for c in event_cols]

        weekly_data = joined_data.groupby(*group_cols).agg(
            F.sum('sales').cast('float').alias('sales'),
            # Price is static per item per wm_yr_wk in M5, so first() is used
            F.first('sell_price').cast('float').alias('sell_price'), 
            # Sum the SNAP flags to get the count of SNAP days in that week (0 to 7)
            F.sum(F.col('snap_CA')).cast('byte').alias('snap_CA_days'),
            F.sum(F.col('snap_TX')).cast('byte').alias('snap_TX_days'),
            F.sum(F.col('snap_WI')).cast('byte').alias('snap_WI_days'),
            F.max('event_flag').cast('byte').alias('event_flag'),
            *event_aggs
        )
        return weekly_data
    
    def leading_zero_sales_removal(self,joined_data):
        window_spec = (Window.partitionBy(self.filter_map['key_col']).orderBy(self.filter_map['yearweek_col']).rowsBetween(Window.unboundedPreceding,Window.currentRow))
        joined_data = joined_data\
                .withColumn('cumsum_sales',F.sum(F.coalesce(F.col(self.filter_map['sales_col']), F.lit(0))).over(window_spec))\
                .filter(F.col("cumsum_sales") > 0)\
                .drop("cumsum_sales")
        return joined_data
    
    def process(self):
        transformed_data = self.transform_data()
        aggregated_data = self.agg_data(transformed_data)
        joined_data = self.leading_zero_sales_removal(aggregated_data).filter(F.col('yearweek')>="201301")
        write_data(joined_data,processed_paths['joined_data'],["state_id"])
        return joined_data




In [0]:
class FeatureEngineering:
    def __init__(self,joined_data,filter_map,m5_config):
        self.joined_data = joined_data
        self.filter_map = filter_map
        self.m5_config = m5_config
        self.id = self.m5_config['id_col']
        self.yearweek = self.m5_config['yearweek_col']
        self.y = self.m5_config['y']
        self.price_col = self.m5_config['price_col']

    
    def creat_lag_features(self,joined_data):
        """computes the sales lags defined in m5_config."""
        lags = self.m5_config['lags_features']

        df = (joined_data[[self.id,self.yearweek,self.y]]
              .drop_duplicates()
              .sort_values(by=[self.id,self.yearweek]))

        for lag in lags:
            df[f"f_{self.y}_lag_{lag}"] = df.groupby(self.id)[self.y].shift(lag)
        
        return joined_data.merge(
            df.drop(columns=[self.y]),
            on=[self.id,self.yearweek],
            how='left'
        )
    
    
    def create_rolling_features(self,joined_data):
        """computes the rolling features"""

        rolling_features = self.m5_config['rolling_features']

        df = (joined_data[[self.id,self.yearweek,self.y]]
              .drop_duplicates()
              .sort_values(by=[self.id,self.yearweek]))

        #rolling average sales
        for window in rolling_features.get('mean',[]):
            df[f"f_{self.y}_ma{window}"] = (
                df.groupby(self.id)[self.y]
                .transform(lambda x: x.shift(1).rolling(window=window).mean())
            )
        
        #rolling std
        for window in rolling_features.get('std',[]):
            df[f"f_{self.y}_rolling_std{window}"] = (
                df.groupby(self.id)[self.y]
                .transform(lambda x: x.shift(1).rolling(window=window).std())
            )

        #rolling max
        for window in rolling_features.get('max',[]):
            df[f"f_{self.y}_rolling_max{window}"] = (
                df.groupby(self.id)[self.y]
                .transform(lambda x: x.shift(1).rolling(window=window).max())
            )

        #rolling min
        for window in rolling_features.get('min',[]):
            df[f"f_{self.y}_rolling_min{window}"] = (
                df.groupby(self.id)[self.y]
                .transform(lambda x: x.shift(1).rolling(window=window).min())
            )

        return joined_data.merge(
            df.drop(columns=[self.y]),
            on=[self.id,self.yearweek],
            how='left'
        )
    
    
    def create_price_features(self,joined_data):
        """computes derived price features, price_change_pct, price_lag1"""

        df = (joined_data[[self.id,self.yearweek,self.price_col]]
              .drop_duplicates()
              .sort_values(by=[self.id,self.yearweek]))

        # Forward fill price within each series.
        # Do not backward fill because that can introduce future information.
        df[self.price_col] = (
            df.groupby(self.id)[self.price_col]
            .ffill()
        )

        df['f_price_change_pct'] = (
            df.groupby(self.id)[self.price_col]
            .pct_change()
            .replace([np.inf, -np.inf], np.nan)
        )
        
        df['f_price_lag1'] = (
            df.groupby(self.id)[self.price_col]
            .shift(1)
        )
        
        return joined_data.drop(columns=[self.price_col]).merge( df,
            on=[self.id,self.yearweek],
            how='left'
        )
    
    
    def create_calendar_features(self,joined_data):
        """computes calendar features for weekly forecasting"""

        df = (
            joined_data[[self.id,self.yearweek,self.y]]
            .drop_duplicates()
            .sort_values(by=[self.id,self.yearweek])
            )

        # Convert YYYYWW string into ISO week date
        df['f_cal_date'] = pd.to_datetime(
            df[self.yearweek].astype(str) + '1',
            format='%G%V%u'
            )

        df['f_cal_weekofyear'] = (
            df['f_cal_date'].dt.isocalendar().week.astype(int)
            )

        df['f_cal_month'] = (
            df['f_cal_date'].dt.month.astype(int)
            )

        df['f_cal_quarter'] = (
            df['f_cal_date'].dt.quarter.astype(int)
            )

        df['f_cal_year'] = (
            df['f_cal_date'].dt.isocalendar().year.astype(int)
            )

        df['f_cal_is_month_start'] = (
            df['f_cal_date'].dt.is_month_start.astype(int)
             )

        df['f_cal_is_month_end'] = (
            df['f_cal_date'].dt.is_month_end.astype(int)
            )

        df['f_cal_is_quarter_start'] = (
            df['f_cal_date'].dt.is_quarter_start.astype(int)
        )

        df['f_cal_is_quarter_end'] = (
            df['f_cal_date'].dt.is_quarter_end.astype(int)
        )

        df['f_cal_is_year_start'] = (
            df['f_cal_date'].dt.is_year_start.astype(int)
        )

        df['f_cal_is_year_end'] = (
            df['f_cal_date'].dt.is_year_end.astype(int)
        )

        return joined_data.merge(
            df.drop(columns=[self.y,'f_cal_date']),
            on=[self.id,self.yearweek],
            how='left'
        )    
    
    def create_event_lead_lags(self,joined_data):
        """computes lead and lag features for important events like christmas, thanksgiving,newyear,laborday"""

        event_lead_lags = self.m5_config['event_lead_lags_features']

        df_columns = [self.id,self.yearweek] + event_lead_lags

        df = (joined_data[df_columns]
              .drop_duplicates()
              .sort_values(by=[self.id,self.yearweek]))

        for event in event_lead_lags:

            # Previous week's event flag
            df[f"f_{event}_lead1"] = (
                df.groupby(self.id)[event].shift(-1)
            )

            # Previous week's event flag
            df[f"f_{event}_lag1"] = (
                df.groupby(self.id)[event].shift(1)
            )

        return joined_data.merge(
            df.drop(columns=event_lead_lags),
            on=[self.id,self.yearweek],
            how='left'
        )
    
    
    def create_seasonlity_features(self,joined_data):
        """computes normalized seasonality features at id, item, store and category levels"""

        week_col = 'f_cal_weekofyear'
        item_col = self.m5_config['product_col']
        store_col = self.m5_config['customer_col']
        cat_col = self.m5_config['category_col']

        df = (
            joined_data[
                [
                    self.id,
                    self.yearweek,
                    self.y,
                    item_col,
                    store_col,
                    cat_col,
                    week_col
                ]
            ]
            .drop_duplicates()
            .sort_values(by=[self.id,self.yearweek])
        )

        # 1. ID-level seasonality
        id_seas = (
            df.groupby([self.id,week_col])[self.y]
            .mean()
            .reset_index()
            .rename(columns={self.y:'id_week_mean'})
        )

        id_mean = (
            df.groupby(self.id)[self.y]
            .mean()
            .reset_index()
            .rename(columns={self.y:'id_overall_mean'})
        )

        id_seas = id_seas.merge(
            id_mean,
            on=[self.id],
            how='left'
        )

        id_seas['f_seasonality_id_level'] = (
            id_seas['id_week_mean'] /
            id_seas['id_overall_mean'].replace(0,np.nan)
        )

        id_seas = id_seas[
            [self.id,week_col,'f_seasonality_id_level']
        ]


        # 2. Item-ID level seasonality
        # Aggregated across stores for an item
        item_seas = (
            df.groupby([item_col,week_col])[self.y]
            .mean()
            .reset_index()
            .rename(columns={self.y:'item_week_mean'})
        )

        item_mean = (
            df.groupby(item_col)[self.y]
            .mean()
            .reset_index()
            .rename(columns={self.y:'item_overall_mean'})
        )

        item_seas = item_seas.merge(
            item_mean,
            on=[item_col],
            how='left'
        )

        item_seas['f_seasonality_itemid_level'] = (
            item_seas['item_week_mean'] /
            item_seas['item_overall_mean'].replace(0,np.nan)
        )

        item_seas = item_seas[
            [item_col,week_col,'f_seasonality_itemid_level']
        ]


        # 3. Store-ID level seasonality
        # Store-wide weekly trend
        store_seas = (
            df.groupby([store_col,week_col])[self.y]
            .mean()
            .reset_index()
            .rename(columns={self.y:'store_week_mean'})
        )

        store_mean = (
            df.groupby(store_col)[self.y]
            .mean()
            .reset_index()
            .rename(columns={self.y:'store_overall_mean'})
        )

        store_seas = store_seas.merge(
            store_mean,
            on=[store_col],
            how='left'
        )

        store_seas['f_seasonality_storeid_level'] = (
            store_seas['store_week_mean'] /
            store_seas['store_overall_mean'].replace(0,np.nan)
        )

        store_seas = store_seas[
            [store_col,week_col,'f_seasonality_storeid_level']
        ]


        # 4. Category-ID level seasonality
        # Broad category weekly trend
        cat_seas = (
            df.groupby([cat_col,week_col])[self.y]
            .mean()
            .reset_index()
            .rename(columns={self.y:'category_week_mean'})
        )

        cat_mean = (
            df.groupby(cat_col)[self.y]
            .mean()
            .reset_index()
            .rename(columns={self.y:'category_overall_mean'})
        )

        cat_seas = cat_seas.merge(
            cat_mean,
            on=[cat_col],
            how='left'
        )

        cat_seas['f_seasonality_category_level'] = (
            cat_seas['category_week_mean'] /
            cat_seas['category_overall_mean'].replace(0,np.nan)
        )

        cat_seas = cat_seas[
            [cat_col,week_col,'f_seasonality_category_level']
        ]


        # Merge all seasonal profiles back to the main dataframe
        df = df.merge(
            id_seas,
            on=[self.id,week_col],
            how='left'
        )

        df = df.merge(
            item_seas,
            on=[item_col,week_col],
            how='left'
        )

        df = df.merge(
            store_seas,
            on=[store_col,week_col],
            how='left'
        )

        df = df.merge(
            cat_seas,
            on=[cat_col,week_col],
            how='left'
        )

        return joined_data.merge(
            df[
                [
                    self.id,
                    self.yearweek,
                    'f_seasonality_id_level',
                    'f_seasonality_itemid_level',
                    'f_seasonality_storeid_level',
                    'f_seasonality_category_level'
                ]
            ],
            on=[self.id,self.yearweek],
            how='left'
        )


    def process(self):
        """processes the data for forecasting"""

        joined_data = self.creat_lag_features(self.joined_data)

        joined_data = self.create_rolling_features(joined_data)

        joined_data = self.create_price_features(joined_data)

        joined_data = self.create_calendar_features(joined_data)

        joined_data = self.create_event_lead_lags(joined_data)

        joined_data = self.create_seasonlity_features(joined_data)

        write_data(
            spark.createDataFrame(joined_data),
            processed_paths['FE'],
            [self.m5_config['category_col']]
        )

        return joined_data